## Введение в обработку изображений и свертки

In [ ]:
import torch
import numpy as np
from PIL import Image
from matplotlib import pyplot as plt
from torch import nn  
from torch.nn.functional import conv2d, pad
IMAGE_PATH = "images"

## PIL, numpy, простые операции

Для начала немного поиграем с представлениями изображений. Наиболее удобный способ читать картинки - через pillow

In [ ]:
img = Image.open(f"{IMAGE_PATH}/pacanoids.jpg")

In [ ]:
plt.imshow(img)

Вопрос: как вы думаете, что не так после преобразования изображения в np?

In [ ]:
img_np = np.array(img)
plt.imshow(255 - img_np)
plt.show()

In [ ]:
img_np.shape

Вопрос: Что получится после выполнения следующего кода?

In [ ]:
img_np = img_np[..., [2,1,0]]

Можно рассмотреть каждый канал отдельно

In [ ]:
from matplotlib import cm

# выведите три графика, на каждом только один канал. Чтобы задать схему, используйте colormaps:
# cmap=cm.Blues, etc

Можем кропнуть изображение

In [ ]:
plt.imshow(img.crop(box=(40, 40, 400, 450)))

**Задание**: кропните это изображение в numpy

**Дополнительное задание 1**: напишите функцию, которая будет принимать на вход numpy array с изображением и возвращать случайный кроп этого изображения заданного размера. <br>
Пример: random_crop(img_np, [30, 30]) -> img_np[0:30, 0:30]

**Дополнительное задание 2**: Напишите то же самое, но с PIL.Image


Конечно, можно и инвертировать изображение. Однако, многие продвинутые операции реализованы в модуле ImageChops

In [ ]:
from PIL import ImageChops
img = ImageChops.invert(img)
plt.imshow(img)

К тому же, изображение можно повернуть

In [ ]:
plt.imshow(img.rotate(45))

In [ ]:
img2 = Image.open(f"{IMAGE_PATH}/akakiy.jpeg")
print(img2.size)
plt.imshow(img2)
img2.mode

In [ ]:
img2 = ImageChops.invert(img2.resize((768//3, 512//3)))
plt.imshow(img2)

Мы можем даже вставить одно изображение в другое. Обратите внимание, что вставка производится прямо в объект, так что нужно предварительн сделать копию, если хочется сохранить исходную картинку

In [ ]:
#cat2 = cat2.resize((768//3, 512//3))
new_img = img.copy()
new_img.paste(img2.copy(), [250, 250])

In [ ]:
new_img

**Дополнительное задание**: напишите функцию, которая будет принимать на вход два изображения и добавлять в случайное место на первом второе

## CNN

До сих пор мы работали с изображением как с единым целым: кропали, крутили, инвертировали. Свёрточные сети смотрят на картинку иначе — **локально**, маленьким окошком, которое скользит по всему изображению и в каждой позиции спрашивает: «похоже то, что подо мной, на мой паттерн?». Паттерном может быть край, текстура шерсти, глаз кота — что угодно. Операция «приложить окошко, посчитать отклик, сдвинуться» называется **свёрткой** (convolution), а сети из таких операций — свёрточными (CNN, Convolutional Neural Networks).

В PyTorch свёрточный слой представлен в модуле torch.nn классом Conv2d (есть также Conv1d для сигналов и текстов и Conv3d для видео и объёмов) с параметрами:

- in_channels: количество входных каналов (для RGB-картинки — 3)
- out_channels: количество выходных каналов (по сути — сколько разных фильтров-детекторов мы применяем)
- kernel_size: размер ядра (того самого окошка)
- stride: значение в пикселях, на которое будет сдвинуто ядро на каждом следующем шаге
- padding: значение в пикселях, ширина "рамки", которая будет добавлена вокруг изображения
- padding_mode: как будет заполняться паддинг ('zeros', 'reflect', 'replicate', 'circular')
- dilation: шаг в пикселях, на который будет "разрежен" вход ядра
- groups: сколько независимых групп каналов сворачивать отдельно (об этом ниже)

Разберёмся, что всё это значит.

### Что такое свёртка

Пусть $I$ — изображение (для простоты пока чёрно-белое, т.е. одноканальное), а $K$ — **ядро** (kernel, он же фильтр) размера $k \times k$. Дискретная свёртка записывается так:

$$O(i, j) = \sum_{m=0}^{k-1} \sum_{n=0}^{k-1} I(i+m,\; j+n)\cdot K(m, n)$$

Читается это так: прикладываем ядро к позиции $(i, j)$, поэлементно перемножаем накрытые пиксели с весами ядра, суммируем — получаем одно число $O(i, j)$. Сдвигаем ядро на следующую позицию — получаем $O(i, j+1)$. И так по всей картинке. Результат $O$ называют **картой активаций** (feature map): в тех местах, где паттерн «похож» на ядро, отклик большой, где не похож — маленький.

> Строго говоря, формула выше — это *взаимная корреляция* (cross-correlation): в «настоящей» математической свёртке ядро ещё и разворачивается на 180°. Фреймворки глубокого обучения ядро не разворачивают — веса всё равно обучаются, так что сторону разворота можно зафиксировать раз и навсегда. Все давно называют это свёрткой, и мы будем.

Если у входа несколько каналов (RGB = 3), ядро становится объёмным — $C_{in} \times k \times k$, и сумма в формуле берётся сразу по всем каналам: свёртка «схлопывает» каналы в один. Выходных каналов будет столько, сколько фильтров мы применим (`out_channels`), и у каждого фильтра — свои веса.

Основное преимущество сверток для изображений в том, что они естественным путем позволяют выучить локальные связи: каждый выходной пиксель зависит только от окрестности $k \times k$, а не от всего изображения.  Кот в углу и кот в центре будут найдены одним и тем же фильтром с одними и теми же весами, что позволит не запоминать все возможные позиции.

**Вопрос**: почему для изображений полносвязный слой так невыгоден, а для табличных данных с признаками «возраст, зарплата, город» — вполне себе нормален?

### Иллюстрации: что делают параметры свёртки

Лучшие наглядные анимации — из репозитория [vdumoulin/conv_arithmetic](https://github.com/vdumoulin/conv_arithmetic). Обозначения: синие квадраты — вход, тёмно-синие — ядро, голубые — выход.

**Обычная свёртка** (kernel=3, padding=0, stride=1): ядро шагает по пикселю за раз, из каждого положения рождается один выходной пиксель. Выход меньше входа на $k-1 = 2$:

![no padding, no strides](../images/no_padding_no_strides.gif)

**Паддинг** (kernel=3, padding=1, stride=1): окружаем вход рамкой из нулей — и «same»-свёртка сохраняет размер картинки. Это позволяет строить глубокие сети, не заботясь о сжатии на каждом слое:

![same padding, no strides](../images/same_padding_no_strides.gif)

**Полный паддинг** (kernel=3, padding=2): обратный трюк — выход *больше* входа:

![full padding, no strides](../images/full_padding_no_strides.gif)

**Шаг** (kernel=3, stride=2): ядро перепрыгивает через пиксели, и размер выхода уменьшается примерно в `stride` раз. Так свёртка умеет понижать разрешение — эту роль иногда отдают пулингу, а иногда именно stride-свёртке:

![no padding, strides](../images/no_padding_strides.gif)

**Паддинг + шаг вместе** (kernel=3, padding=1, stride=2) — стандартный строительный блок «уменьшить вдвое, ничего не потеряв по краям»:

![padding + strides](../images/Convolution_arithmetic_-_Padding_strides.gif)

**Дилатация** (kernel=3, dilation=2): между элементами ядра появляются «дырки», ядро 3×3 охватывает область 5×5. Рецептивное поле растёт, а число параметров — нет:

![dilation](../images/Convolution_arithmetic_-_Dilation.gif)

**Вопрос**: какую по размеру область входа «видит» ядро 3×3 с dilation=3? Что произойдёт с охватом, если применить такую свёртку два раза подряд?

Рассмотрим несколько фильтров:

Фильтр улучшения четкости
[[ -1, -1, -1,],
[  -1, 9, -1],
[  -1, -1, -1]]

Фильтр Собеля - для выявления горизонтальных ребер
[[ -1, -2, -1],
[  0,  0,  0],
[  1,  2,  1]]

Он вычитает (со весами) яркость верхней строки из нижней, поэтому реагирует на горизонтальные перепады яркости. Важное свойство: веса в каждом столбце в сумме дают ноль, значит, на однотонном фоне отклик нулевой — «зажигаются» только перепады. Это свойство нам очень пригодится в задании про сумму квадратов: без него на выходе будет не контурное изображение, а просто пересвеченная копия картинки.

**Вопрос**: что нужно изменить для того, чтобы выявлять вертикальные ребра?

Функция ожидает на вход батчи, так что необходимо добавить новое измерение, чтобы получился виртуальный батч из 1 картинки. Кроме того, каналы должны идти сразу после батча (размерность NCHW).

In [ ]:
img_tensor = torch.tensor(img_np, dtype=torch.float)

img_tensor = img_tensor # поменяйте размерности и долбавьте размерность батча
img_tensor.size()

Потренируемся с фильтром Собеля. Нам нужно применить его ко всем каналам входного изображения, так что нужно его размножить.

In [ ]:
kernel = [[ -1, -2, -1],
[  0,  0,  0],
[  1,  2,  1]]
hor_filter = torch.tensor([[kernel, kernel, kernel]], dtype=torch.float)
hor_filter.size()

In [ ]:
hor_result = conv2d(img_tensor, hor_filter).squeeze()
plt.imshow(hor_result)

Задание: Создайте и примените вертикальный фильтр Собеля.

In [ ]:
# создайте ядро 
kernel =  
ver_filter = torch.tensor([[kernel, kernel, kernel]], dtype=torch.float)
ver_filter.size()
ver_result = conv2d(img_tensor, ver_filter).squeeze()
plt.imshow(ver_result)

**Задание.** Объединим результаты, получив представление всех ребер)
Например, это можно сделать, посчитав сумму квадратов результатов. Такая величина называется магнитудой градиента (классический вариант — с корнем: $\sqrt{G_x^2 + G_y^2}$). Благодаря нулевой сумме весов у обоих фильтров фон уходит в ноль, и на выходе должны остаться только контуры.

In [ ]:
img_conv = # сумма квадратов результатов верт. и хор. фильтров
plt.imshow(img_conv)
plt.show()

### Размер выхода

Все параметры выше складываются в одну формулу. Для квадратного входа размера $W$ выход имеет сторону

$$O = \left\lfloor \frac{W + 2P - D \cdot (K - 1) - 1}{S} \right\rfloor + 1$$

где $K$ — размер ядра, $P$ — паддинг, $S$ — stride, $D$ — dilation. Проверьте на анимациях из прошлого раздела: например, $W=5$, $K=3$, $P=0$, $S=1$ даёт $O=3$ — ровно как на первой гифке.

**Вопрос**: при каких $P$, $S$ и $D$ свёртка сохраняет размер (same) для ядра 3×3? А для ядра 7×7?

In [ ]:
def conv_out_size(w, kernel_size, padding=0, stride=1, dilation=1):
    return  # реализуйте формулу рассчета размера

from torch import nn
import pandas as pd

rows = []
for kernel_size, padding, stride, dilation in [
    (3, 0, 1, 1),  # «ванильная» свёртка
     #TODO ,  # same: размер не меняется
    (3, 2, 1, 1),  
    (3, 0, 2, 1),  
    (3, 1, 2, 1),  
    #TODO ,  # 7x7 + same: размер не меняется
    (3, 0, 1, 2),  # dilation=2 
]:
    w = 32
    out = conv_out_size(w, kernel_size, padding, stride, dilation)
    layer = nn.Conv2d(3, 16, kernel_size, padding=padding, stride=stride, dilation=dilation)
    assert layer(torch.randn(1, 3, w, w)).shape[-1] == out 
    
    rows.append({"kernel_size": kernel_size, "padding": padding, "stride": stride,
                 "dilation": dilation, f"выход (вход {w})": out})

pd.DataFrame(rows)

### Страйд, паддинг и дилатация: сравниваем на живой картинке

Мы уже видели, как параметры свёртки меняют размер выхода. Теперь посмотрим, как они меняют рецептивное поле и поле признаков. 

In [ ]:
img = Image.open(f"{IMAGE_PATH}/akakiy.jpeg")
img_gray = torch.tensor(np.array(img), dtype=torch.float).mean(dim=-1).unsqueeze(0).unsqueeze(0)
print(img_gray.shape)

plt.imshow(img_gray.squeeze(), cmap="gray")

Для начала посмотрим, как будет выглядеть результат с разным паддингом:

In [ ]:
# одно и то же усредняющее ядро (box blur) с разным паддингом
box = torch.ones(1, 1, 9, 9) / 81

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, p in zip(axes, [0, 4, 16]):
    res = conv2d(img_gray, box, padding=p)
    ax.imshow(res.squeeze(), cmap="gray")
    ax.set_title(f"padding={p}: выход {res.shape[-2]}x{res.shape[-1]}")
fig.suptitle(f"Сравнение padding (вход {img_gray.shape[-2]}x{img_gray.shape[-1]})")

Также посмотрим, каким будет выход для свертки с разным шагом:

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, s in zip(axes, [# задайте stride]):
    res = conv2d(img_gray, box, padding=1, stride=s)
    ax.imshow(res.squeeze(), cmap="gray")
    ax.set_title(f"stride={s}: выход {res.shape[-2]}x{res.shape[-1]}")
fig.suptitle(f"stride: вход {img_gray.shape[-2]}x{img_gray.shape[-1]}")

#### Дилатация

Ядро 3×3 с dilation=$D$ охватывает квадрат со стороной $K + (K-1)(D-1) = 2D+1$, но весов в нём по-прежнему девять. Верхний ряд ниже показывает, какие именно пиксели входа использует такое ядро, а нижний — что получается, если применить его к изображению: с ростом дилатации один и тот же фильтр Собеля начинает реагировать на всё более крупные вертикальные перепады. Это справедливо и для свертки общего вида - чем больше дилация, тем больше рецептивное поле фильтра без увеличения числа весов.

In [ ]:
sobel_v = torch.tensor([[1., 0., -1.],
                        [2., 0., -2.],
                        [1., 0., -1.]]).unsqueeze(0).unsqueeze(0)

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for col, d in enumerate([1, 2, 4]):
    mask = torch.zeros(2 * d + 1, 2 * d + 1)
    mask[::d, ::d] = 1
    axes[0, col].imshow(mask, cmap="Blues")
    axes[0, col].set_title(f"dilation={d}: охват {2 * d + 1}x{2 * d + 1}")

    res = conv2d(img_gray, sobel_v, padding=d, dilation=d)
    axes[1, col].imshow(res.squeeze().abs(), cmap="gray")
    axes[1, col].set_title(f"Собель по x, dilation={d}: {res.shape[-2]}x{res.shape[-1]}")
fig.tight_layout()


#### Depth-wise convolution
До сих пор мы работали с фильтрами (или входами), которые не требовали многоканального выхода.

Если мы попытаемя сделать также с фильтром усиления четкости, то получим одноканальное изображение. Это не очень хорошо. Мы хотим получить такую же картинку, как на входе, но четче. Простейший вариант - применить фильтр к каждому каналу отдельно. Попробуем сделать это.


In [ ]:
sharp_kernel = [[ -1, -1, -1,],
[  -1, 9, -1],
[  -1, -1, -1]]
weights = torch.tensor(sharp_kernel, dtype=torch.float).unsqueeze(0).unsqueeze(0)
res = torch.stack([conv2d(img_tensor[:, i,...], weights, padding=1) for i in range(3)], dim=1)

In [ ]:
plt.imshow(res.squeeze().permute(1,2,0).clip(0, 255).to(torch.int))

Однако, это не единственный способ. Можно воспользоваться depth-wise convolution. Обычная свертка делается сразу по всем каналам и мешает их как угодно. depth-wise convolution же, в свою очередь, делает следующие три шага:
- Разбивает фильтр и и вход на каналы
- Сворачивает каждый канал отдельно соответствующим фильтром
- Конкатенирует результат

В torch.nn это уже реализовано в conv2d (Conv2d) с помощью параметра groups. На самом деле, эта возможность достаточно мощная. Она позволяет как уменьшить число параметров, сохраняя точность, так и применять группы сверток параллельно. Это очень полезно - чуть позже мы узнаем, какая модель, основанная на этой простой идее, когда-то порвала все лидерборды.

**Задание**: Примените ядро усиления четкости к трем каналам картинки с использованием параметра groups.

In [ ]:
sharp_kernel = [[ -1, -1, -1,],
[  -1, 9, -1],
[  -1, -1, -1]]
weights = torch.tensor([sharp_kernel, sharp_kernel,sharp_kernel], dtype=torch.float)
# вес depthwise-свёртки (groups=3) должен иметь форму
# [out_channels, in_channels // groups, kH, kW] = [3, 1, 3, 3],
# поэтому размерность «одиночного канала» вставляем на ВТОРУЮ позицию, а не в конец:
weights = # поправьте weights

In [ ]:
sharp_result = conv2d(img_tensor, weights, padding=1, groups=3)
plt.imshow(sharp_result.squeeze().permute(1,2,0).clip(0, 255).to(torch.int))

**Задание**: Реализуйте и примените случайный фильтр 

**Дополнительное задание**: Реализуйте и примените гауссовский фильтр. Этот фильтр дожен приводить к размытию входного изображения

Рассмотрим теперь слой nn.Conv2d. Синтаксис при его инстанциировании практически такой же, как у функции. Сверточный слой обучаемый, матрица весов инициализируется с помощью стратегии инициализации. Пользователю же необходимо задать входное и выходное число каналов и размер ядра.


In [ ]:
from torch import nn

conv = nn.Conv2d(in_channels=3, out_channels=3, kernel_size=3, padding=1)

In [ ]:
conv.weight.shape

Посмотрим на результат:

In [ ]:
sharp_result = conv(img_tensor)
plt.imshow(sharp_result.squeeze().permute(1,2,0).clip(0, 255).to(torch.int))

In [ ]:
from torch import nn

conv = nn.Conv2d(in_channels=3, out_channels=3, kernel_size=3, padding=1, groups=3)
result = conv(img_tensor)
plt.imshow(result.squeeze().permute(1,2,0).clip(0, 255).to(torch.int))

Как видите, с учетом случайной инициализации мы получили достаточно разные результаты.

#### Сколько весов в обычной, групповой и depthwise-свёртке

Про параметр `groups` мы уже говорили выше. Число весов свёртки считается так:

$$\text{params} = C_{out} \cdot \frac{C_{in}}{G} \cdot K \cdot K$$

При $G=1$ это обычная свёртка: каждый выходной канал смотрит на *все* входные. При $G=C_{in}$ получается depthwise: каждый канал сворачивается со своим маленьким фильтром независимо. Промежуточные $G$ — компромисс. Посчитаем на живых слоях ($C_{in}=C_{out}=8$, $K=3$):

In [ ]:
from torch import nn
import pandas as pd

rows = []
for groups, name in [(1, "обычная (G=1)"), (2, "G=2"), (4, "G=4"), (8, "depthwise (G=C_in=8)")]:
    layer = nn.Conv2d(8, 8, kernel_size=3, padding=1, groups=groups, bias=False)
    rows.append({"режим": name, "число весов": sum(p.numel() for p in layer.parameters())})

pd.DataFrame(rows)

Экономия в весах достигает х8 раз (что довольно много). Однако, если полностью разделить все преобразования, получить прищнаки из взаимодействия каналов мы не сможем. Как эта проблема решается? Достаточно прямолинейно - перемешаем все каналы уже после свертки, отдельно. Простейший способ - применить свертку 1х1. Число весов при этом практически не увеличивается (посчитайте, насколько их больше).

На этой идее (depthwise 3×3 + смешивающая каналы свёртка 1×1, вместе — *depthwise separable convolution*) построены MobileNet и её наследники — модели для мобильных устройств. 

#### Depthwise separable convolution

Доведём идею `groups` до конца. Обычная свёртка 3×3 делает сразу две работы: ищет паттерны **по пространству** и смешивает **каналы**. Depthwise separable convolution раскладывает её на две простые:

1. **depthwise 3×3** — каждый канал сворачивается со своим фильтром, каналы не общаются: $C_{in} \cdot K^2$ весов;
2. **pointwise 1×1** — обучаемое перемешивание каналов в каждой точке: $C_{in} \cdot C_{out}$ весов.

Против обычной свёртки с её $C_{out} \cdot C_{in} \cdot K^2$ весами выигрыш равен

$$\frac{C_{in} K^2 + C_{in} C_{out}}{C_{out} C_{in} K^2} = \frac{1}{C_{out}} + \frac{1}{K^2}$$

— то есть уже $\approx 1/9$ от стоимости обычной свёртки даже при огромном числе каналов. Проверим на живых слоях:

In [ ]:
# сравниваем число весов при разных размерах каналов
rows = []
for c_in, c_out in [(8, 16), (32, 64), (256, 256), (512, 512)]:
    regular = nn.Conv2d(c_in, c_out, kernel_size=3, bias=False)
    separable = nn.Sequential(
        nn.Conv2d(c_in, c_in, kernel_size=3, groups=c_in, bias=False),  # depthwise 3x3
        nn.Conv2d(c_in, c_out, kernel_size=1, bias=False),              # pointwise 1x1
    )
    n_reg = sum(p.numel() for p in regular.parameters())
    n_sep = sum(p.numel() for p in separable.parameters())
    rows.append({"C_in -> C_out": f"{c_in} -> {c_out}", "обычная 3x3": n_reg,
                 "separable (dw+pw)": n_sep, "выигрыш, раз": round(n_reg / n_sep, 1)})

pd.DataFrame(rows)

In [ ]:
torch.manual_seed(42)
cat3 = torch.tensor(np.array(cat), dtype=torch.float).permute(2, 0, 1).unsqueeze(0)

regular = nn.Conv2d(3, 8, kernel_size=3, padding=1, bias=False)
separable = nn.Sequential(
    nn.Conv2d(3, 3, kernel_size=3, padding=1, groups=3, bias=False),
    nn.Conv2d(3, 8, kernel_size=1, bias=False),
)

n_reg = sum(p.numel() for p in regular.parameters())
n_sep = sum(p.numel() for p in separable.parameters())

fig, axes = plt.subplots(2, 8, figsize=(16, 4.6))
for row, (name, layer, n) in enumerate([
        (f"обычная 3x3: {n_reg} весов", regular, n_reg),
        (f"depthwise separable: {n_sep} весов", separable, n_sep)]):
    maps = layer(cat3).detach().squeeze()  # [8, H, W]
    for fmap in range(8):
        axes[row, fmap].imshow(maps[fmap].abs(), cmap="gray")
        axes[row, fmap].set_xticks([])
        axes[row, fmap].set_yticks([])
    axes[row, 3].set_title(name, fontsize=12, pad=10)
fig.suptitle(f"Карты признаков размера {tuple(maps.shape[-2:])}", y=1.02)
fig.tight_layout()

На случайных весах картинки карт признаков, конечно, выглядят похоже — смысл не в этом, а в экономии: при $C_{in}=3 \to C_{out}=8$ выигрыш пока скромный ($4.2\times$), но в реальных сетях, где каналов сотни, — гораздо внушительнее. Именно на этом компромиссе «чуть менее выразительный слой, зато в разы дешевле» построены MobileNet и её наследники: свёртка становится настолько дешёвой, что можно позволить себе гораздо более глубокую сеть при том же бюджете вычислений.

**Вопрос**: у вас есть свёртка 3×3, меняющая 256 каналов на 256. Сколько весов вы сэкономите, заменив её на depthwise separable? А сколько — если заменить её на свёртку 1×1? Какая замена «дешевле» и почему тем не менее нужны обе?

# Бонус 1. Свёртка своими руками

`conv2d` из pytorch — оптимизированная операция, внутри неё ничего не скользит: всё считается матричными умножениями. Чтобы почувствовать механику, реализуем свёртку в лоб — вложенными циклами, ровно по формуле выше.

In [ ]:
def conv2d_manual(x, kernel):
    """Простейшая реализация свёртки

    x: [1, 1, H, W], kernel: [1, 1, kH, kW]
    """
    _, _, h, w = x.shape
    _, _, kh, kw = kernel.shape
    out = torch.zeros(h - kh + 1, w - kw + 1)
    for i in range(out.shape[0]):
        for j in range(out.shape[1]):
            # вырезаем окно kh x kW, начиная с позиции (i, j)
            patch = # TODO
            # поэлементно умножаем на ядро и суммируем
            out[i, j] = # TODO
    return out

x = torch.randn(1, 1, 8, 8)
kernel = torch.randn(1, 1, 3, 3)

manual = conv2d_manual(x, kernel)
fast = conv2d(x, kernel)
print("форма выхода:", manual.shape)
print("максимальное расхождение с F.conv2d:", (manual - fast).abs().max().item())

## Свёртка как матричное умножение: im2col + GEMM
Обычно же свертку считают с помощью матричного умножения - это гораздо эффективнее (BLAS на CPU, cuDNN/oneDNN на GPU). Можно было бы, конечно, попытаться представить свертку в "длинной" форме, но хитрость, превращающая свёртку в одно матричное умножение, немного иная - это **im2col** (image to columns):

1. Вырезаем из картинки окна во всех возможных положениях ядра и разворачиваем каждое в столбец. Получается матрица патчей $P$ размера $(C_{in} \cdot k \cdot k) \times L$, где $L = H_{out} \cdot W_{out}$ — число положений ядра.
2. Разворачиваем веса свёртки в матрицу $W$ размера $C_{out} \times (C_{in} \cdot k \cdot k)$.
3. Результат свёртки — это просто $W \cdot P$ размера $C_{out} \times L$. Осталось свернуть его обратно в карту $C_{out} \times H_{out} \times W_{out}$.

Вся арифметика теперь живёт в одном умножении; сборка патчей — это перемещение данных, а не вычисления. Примерно так (с вариациями) свёртку и считают внутри глубоких фреймворков.

In [ ]:
def im2col(x, kernel_size):
    """Разворачивает все окна ядра в столбцы матрицы.

    x: [1, C_in, H, W] -> [C_in * k * k, H_out * W_out]
    """
    c, h, w = x.shape[1:]
    k = kernel_size
    out_h, out_w = h - k + 1, w - k + 1
    patches = torch.zeros(c * k * k, out_h * out_w)
    col = 0
    for i in range(out_h):
        for j in range(out_w):
            # окно [C_in, k, k], начиная с позиции (i, j), вытягиваем в столбец
            patches[:, col] = # TODO
            col += 1
    return patches


def conv2d_matmul(x, kernel):
    """Свёртка через одно матричное умножение.

    x: [1, C_in, H, W], kernel: [C_out, C_in, k, k]
    """
    c_out, c_in, k, _ = kernel.shape
    patches = im2col(x, k)                          # [C_in*k*k, L]
    weights = kernel # преобразуйте шейпы           # [C_out, C_in*k*k]
    out = # TODO проведите умножение 
    out_h, out_w = x.shape[-2] - k + 1, x.shape[-1] - k + 1
    return out.reshape(c_out, out_h, out_w).unsqueeze(0)

x = torch.randn(1, 3, 8, 8)
kernel = torch.randn(4, 3, 3, 3)  # 4 фильтра, каждый видит все 3 канала

print("форма выхода:", conv2d_matmul(x, kernel).shape)
print("max |diff| с F.conv2d:", (conv2d_matmul(x, kernel) - conv2d(x, kernel)).abs().max().item())

В pytorch im2col уже есть из коробки — это функция unfold. Сравним скорость наших реализаций с коробочной

In [ ]:
import time
from torch.nn.functional import unfold

print("unfold:", unfold(x, kernel_size=3).shape)  # [1, C*k*k, L] — те же патчи

big = torch.randn(1, 1, 128, 128)
kern = torch.randn(1, 1, 3, 3)
kern_mat = kern.reshape(1, 9)

def bench(fn, n_runs=3):
    times = []
    for _ in range(n_runs):
        t0 = time.perf_counter()
        fn()
        times.append(time.perf_counter() - t0)
    return min(times)  # лучший из трёх — чтобы сгладить шум


def run_manual():
    conv2d_manual(big, kern)


def run_im2col_loops():
    patches = im2col(big, 3)          # сборка патчей циклами
    return kern_mat @ patches


def run_unfold():
    patches = unfold(big, kernel_size=3).squeeze(0)  # та же сборка, но векторизованно
    return kern_mat @ patches


def run_fast():
    return conv2d(big, kern)


for name, fn in [("циклы (conv2d_manual)", run_manual),
                 ("im2col циклами + @", run_im2col_loops),
                 ("F.unfold + @", run_unfold),
                 ("F.conv2d", run_fast)]:
    print(f"{name:22s}: {bench(fn) * 1000:9.3f} мс")

Даже при том, что мы написали im2col на питоне, мы уже выиграли достаточно много.

# Бонус 2: транспонированная свёртка

А бывает ли свёртка, которая *увеличивает* картинку? Да — транспонированная (transposed convolution, в народе deconvolution). Обычная свёртка «собирает» несколько входных пикселей в один выходной; транспонированная работает в противоположную сторону — «разбрасывает» каждый входной пиксель по нескольким выходным. Такие слои нужны везде, где надо перейти от маленьких карт активаций к большим: в сегментации, генераторах GAN, апсемплинге.

![transposed convolution](../images/Convolution_arithmetic_-_Padding_strides_transposed.gif)

![transposed convolution, odd](../images/Convolution_arithmetic_-_Padding_strides_odd_transposed.gif)


Механика транспонированной свертки следующая:

- обычная свёртка **собирает**: каждый выходной пиксель — сумма входных пикселей, накрытых ядром в одном его положении;
- транспонированная **разбрасывает**: каждый входной пиксель умножается на ядро целиком, и получившийся маленький «штамп» кладётся в соответствующее место выхода. Штампы соседних входных пикселей перекрываются, и там, где они пересекаются, вклады суммируются.

Stride в данном случае — это шаг, с которым штампы ложатся на выход: при `stride=2` штамп от следующего входного пикселя ставится на две позиции дальше — вот и рост разрешения.


> **Почему «deconvolution» — некорректное название.** $T^\top \neq T^{-1}$: транспонированная свёртка не обращает свёртку и ничего не «восстанавливает». Свёртка теряет информацию (много входов → один выход), честного обратного оператора не существует. На деле слой — просто обучаемый апсемплер.

In [ ]:
# 1D: свёртка с ядром [10, 20, 30] — это умножение на матрицу T размера [4, 6]
kernel_1d = torch.tensor([10., 20., 30.])
T = torch.zeros(4, 6)
for i in range(4):
    T[i, i:i + 3] = kernel_1d

x = torch.arange(1., 7.)   # «большой» вход длины 6
y = torch.arange(1., 5.)   # «маленький» вход длины 4

print("свёртка            T    @ x :", (T @ x).tolist())
print("транспонированная  T.T  @ y :", # Сделайте транспонированную свертку)
print("F.conv_transpose1d          :", nn.functional.conv_transpose1d(
    y.view(1, 1, -1), kernel_1d.view(1, 1, -1)).squeeze().tolist())

In [ ]:
def conv_transpose2d_manual(x, kernel):
    _, _, h, w = x.shape
    kh, kw = kernel.shape
    out = torch.zeros(1, 1, (h - 1) + kh, (w - 1) + kw)
    for i in range(h):
        for j in range(w):
            # пиксель (i, j) кладёт ядро, умноженное на его значение, в позицию (i, j) выхода
            out[0, 0, i:i + kh, j:j + kw] += # todo, почему плюс, а не присваивание?
    return out

x_tiny = torch.tensor([[1., 0.], [0., 5.]])
k_tiny = torch.tensor([[1., 2., 1.], [2., 4., 2.], [1., 2., 1.]]) / 4

out = conv_transpose2d_manual(x_tiny.view(1, 1, 2, 2), k_tiny)
print("вход 2x2 -> выход", tuple(out.shape[-2:]))
print("совпадает с F.conv_transpose2d:", torch.allclose(
    out, nn.functional.conv_transpose2d(x_tiny.view(1, 1, 2, 2), k_tiny.view(1, 1, 3, 3))))

# сколько штампов накрыло каждую клетку выхода
counts = conv_transpose2d_manual(torch.ones(1, 1, 2, 2), torch.ones(3, 3)).squeeze()

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, m, title in [(axes[0], x_tiny, "вход 2x2"),
                     (axes[1], out.squeeze(), "выход 4x4: сумма штампов"),
                     (axes[2], counts, "сколько штампов накрыло клетку")]:
    ax.imshow(m, cmap="Blues")
    ax.set_title(title)
    ax.set_xticks([]); ax.set_yticks([])
    for i in range(m.shape[0]):
        for j in range(m.shape[1]):
            ax.text(j, i, f"{m[i, j]:.1f}", ha="center", va="center", fontsize=9)
fig.tight_layout()

#### Размер выхода

При stride $S$ соседние штампы шагают на $S$ позиций, поэтому для входа размера $W$:

$$O = S \cdot (W - 1) + K - 2P + OP$$

где $OP$ (`output_padding`) — добавка в один пиксель, которой иногда не хватает, чтобы получить нужный размер: она влияет только на размер выхода и ни в каких вычислениях не участвует.  

**Задание**: проверьте вручную, что $K=4$, $S=2$, $P=1$, $OP=0$ тоже даёт $O = 2W$ — этот вариант без `output_padding` встречается, например, в генераторах DCGAN.

In [ ]:
up = nn.ConvTranspose2d(1, 1, kernel_size=3, stride=2, padding=1, output_padding=1)
small = torch.randn(1, 1, 16, 16)
print("вход:", small.shape, "-> выход:", up(small).shape)  # каждая сторона удвоилась

#### Шахматные артефакты

Если какие-то выходные пиксели получают два вклада, а их соседи один, — на выходе возникает регулярная решётка (знаменитые checkerboard artifacts в генерациях GAN). Посмотрите на третью картинку выше: центр накрыт дважды, а углы — по одному разу.

Сделать $K$ кратным $S$ — полдела: число вкладов выравнивается (при $k{=}4, s{=}2$ каждый внутренний пиксель получает ровно два), но чётные и нечётные выходные пиксели складывают при этом разные веса ядра: одни — с чётных позиций $\{0, 2\}$, другие — с нечётных $\{1, 3\}$. На случайных весах эти суммы почти наверняка различаются, и слабая решётка остаётся. Эти артефакты могут пропасть при обьучении, но можно использовать гладкое ядро с самого начала: например, билинейное $\frac{1}{8}[1, 3, 3, 1]$ — у него чётные и нечётные суммы весов совпадают, и решётки нет даже без обучения.

Сравним разные способы апсемплинга:

In [ ]:
torch.manual_seed(0)

up_bad = nn.ConvTranspose2d(1, 1, kernel_size=3, stride=2, padding=1, output_padding=1, bias=False)
up_good = nn.ConvTranspose2d(1, 1, kernel_size=4, stride=2, padding=1, bias=False)

# то же k=4, но ядро заранее гладкое, как у билинейной интерполяции
bilinear1d = torch.tensor([1., 3., 3., 1.]) / 8
up_smooth = nn.ConvTranspose2d(1, 1, kernel_size=4, stride=2, padding=1, bias=False)
up_smooth.weight.data = torch.outer(bilinear1d, bilinear1d).view(1, 1, 4, 4)

resize_conv = nn.Sequential(
    nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False),
    nn.Conv2d(1, 1, kernel_size=3, padding=1, bias=False),
)

demo = up_good(torch.ones(1, 1, 8, 8)).detach().squeeze()
demo_s = up_smooth(torch.ones(1, 1, 8, 8)).detach().squeeze()
print("вход единичный, фрагмент строки выхода:")
print("  случайные веса:    ", [f"{v:.2f}" for v in demo[4, 4:8].tolist()])
print("  билинейное ядро:   ", [f"{v:.2f}" for v in demo_s[4, 4:8].tolist()])

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
variants = [
    (up_bad(img_gray).detach(), "k=3, s=2"),
    (up_good(img_gray).detach(), "k=4, s=2"),
    (up_smooth(img_gray).detach(), "k=4, s=2, билинейное ядро"),
    (resize_conv(img_gray).detach(), "bilinear resize + свёртка"),
]
for ax, (img, title) in zip(axes, variants):
    ax.imshow(img.squeeze()[280:400, 480:640], cmap="gray")
    ax.set_title(title)
    ax.set_xticks([]); ax.set_yticks([])
fig.tight_layout()

# энергия показывает неравномерность случайной решетки 
checker = torch.tensor([[1., -1.], [-1., 1.]]).view(1, 1, 2, 2)
for img, title in variants:
    energy = conv2d(img, checker).pow(2).mean().item()
    print(f"{title:38s} энергия решётки: {energy:9.2f}")

 Вторая часть занятия пройдет в отдельном ноутбуке, simple_cnn_model